# Automated Reasoning Evaluation

This notebook demonstrates how to evaluate **Automated Reasoning (AR) Checks** in Amazon Bedrock Guardrails.

AR Checks verify LLM outputs against formal policy rules using a two-step pipeline:
1. **Translation** (LLM-based): Natural language → logical formulas
2. **Verification** (SMT solver): Check logical formulas against policy rules

This produces one of **7 verdict types**, ordered by severity (worst → best):

| Verdict | Meaning | Actionable? |
|---------|---------|-------------|
| `TRANSLATION_AMBIGUOUS` | Multiple interpretations of the input | Improve variable descriptions or ask user to clarify |
| `IMPOSSIBLE` | Contradictory premises detected | Review input for logical inconsistencies |
| `INVALID` | Claim violates at least one policy rule | Block or rewrite the response |
| `SATISFIABLE` | Some interpretations are valid, some are not | Add constraints or ask for clarification |
| `VALID` | Claim satisfies all translated policy rules | Serve the response |
| `TOO_COMPLEX` | SMT solver timed out | Simplify the claim or split into parts |
| `NO_TRANSLATIONS` | No policy rules matched the input | Out of scope for this policy |

> **Time:** ~20 minutes
>
> **Deep dives available:**
> - `04-02-07a` — Document preprocessing pipeline (standalone, ~30 min)
> - `04-02-07b` — Comprehensive evaluation with full test suite (~30 min)
> - `04-02-07c` — Multi-guardrail architecture + fidelity optimization (~45 min, requires MCP)

In [ ]:
!uv venv .venv 2>/dev/null; source .venv/bin/activate && uv pip install -q -r requirements.txt

In [ ]:
import json
import time
import re
import uuid
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from botocore.config import Config

# AR finding types in severity order (worst → best)
FINDING_TYPES = [
    'translationAmbiguous', 'impossible', 'invalid',
    'satisfiable', 'valid', 'tooComplex', 'noTranslations'
]
SEVERITY_ORDER = {ft: i for i, ft in enumerate(FINDING_TYPES)}

# Plot styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
config = Config(retries={'max_attempts': 3})
bedrock_client = boto3.client('bedrock', config=config)
bedrock_runtime_client = boto3.client('bedrock-runtime', config=config)

try:
    bedrock_client.list_guardrails(maxResults=1)
    print("Bedrock access verified")
except Exception as e:
    print(f"ERROR: Cannot access Bedrock. Check your AWS credentials.\n{e}")

In [ ]:
RULES_PATH = 'data/housing_code_structured_rules.md'

with open(RULES_PATH, 'r') as f:
    structured_rules = f.read()

rule_count = structured_rules.count('RULE [')
definition_count = structured_rules.count('DEFINITION:')
exception_count = structured_rules.count('EXCEPTION')
print(f"Loaded structured rules: {rule_count} rules, {definition_count} definitions, {exception_count} exceptions")
print(f"Document size: {len(structured_rules):,} characters")

## I. Create an Automated Reasoning Policy

We create AR policies from the structured rules document. Each policy covers a domain (chapter) of the housing code. We select the top 2 domains by rule count — the maximum allowed per guardrail.

> **Key insight:** Variable descriptions are *the single most important determinant of translation accuracy*. Good descriptions include synonyms, units, boundary conditions, and plain language explanations.

In [ ]:
blocks = re.split(r'\n---\n', structured_rules)
shared_defs = blocks[0].strip()

chapter_blocks = {}
for block in blocks[1:]:
    sections = re.findall(r'§\s*(\d{3,})', block)
    if not re.findall(r'RULE\s*\[', block):
        continue
    if sections:
        section_num = int(Counter(sections).most_common(1)[0][0])
        chapter_num = section_num // 100
        key = f"CHAPTER_{chapter_num}"
        chapter_blocks.setdefault(key, []).append(block.strip())

domain_docs = {}
for chapter_key, chapter_block_list in chapter_blocks.items():
    domain_docs[chapter_key] = f"{shared_defs}\n\n---\n\n" + "\n\n---\n\n".join(chapter_block_list)

domain_rule_counts = {
    k: sum(b.count('RULE [') for b in v)
    for k, v in chapter_blocks.items()
    if sum(b.count('RULE [') for b in v) > 0
}
top_domains = sorted(domain_rule_counts.items(), key=lambda x: -x[1])[:2]

print(f"Found {len(domain_rule_counts)} domains with rules")
print(f"\nTop 2 domains selected for guardrail:")
for domain, count in top_domains:
    print(f"  {domain}: {count} rules")
if len(domain_rule_counts) > 2:
    print(f"\n  (Remaining {len(domain_rule_counts) - 2} domains available in notebook 07c)")

In [ ]:
unique_id = str(uuid.uuid4())[:8]
policies = []

for domain_key, rule_count in top_domains:
    policy_name = f"housing-code-{domain_key.lower()}-{unique_id}"
    source_text = domain_docs[domain_key]

    print(f"\nCreating policy: {policy_name} ({rule_count} rules)...")

    create_resp = bedrock_client.create_automated_reasoning_policy(
        name=policy_name,
        policyType='INGEST_CONTENT',
        policyDefinition={
            'ingestContent': {
                'sourceDocuments': [{
                    'content': {'text': source_text},
                    'contentType': 'text/plain'
                }],
                'documentDescription': (
                    "San Francisco Housing Code regulations with if-then rules. "
                    "Ignore lines containing '(omitted)' or '(portions omitted)' \u2014 "
                    "these are editorial markers, not rules. Do not create rules about "
                    "omitted sections. Only extract from sections with substantive "
                    "verifiable requirements. Ignore page headers, footers, and "
                    "editorial commentary."
                )
            }
        }
    )

    policy_arn = create_resp['policyArn']
    print(f"  Policy ARN: {policy_arn}")

    bedrock_client.start_automated_reasoning_policy_build_workflow(policyArn=policy_arn)
    print(f"  Build started, waiting for completion...")

    while True:
        status_resp = bedrock_client.get_automated_reasoning_policy(policyArn=policy_arn)
        status = status_resp.get('status', 'UNKNOWN')
        if status in ('ACTIVE', 'FAILED'):
            break
        time.sleep(10)

    if status == 'ACTIVE':
        build_assets = status_resp.get('policyDefinition', {})
        actual_rules = len(build_assets.get('smtPolicyDefinition', {}).get('rules', []))
        actual_vars = len(build_assets.get('smtPolicyDefinition', {}).get('variables', []))
        print(f"  Build complete: {actual_rules} rules, {actual_vars} variables")
    else:
        print(f"  Build FAILED: {status_resp.get('failureReasons', 'unknown')}")

    policies.append({
        'name': policy_name,
        'domain': domain_key,
        'policy_arn': policy_arn,
        'rule_count': rule_count,
        'status': status
    })

print(f"\n{'='*60}")
print(f"Created {len(policies)} policies")

In [ ]:
versioned_arns = []
for p in policies:
    if p['status'] != 'ACTIVE':
        continue
    version_resp = bedrock_client.create_automated_reasoning_policy_version(
        policyArn=p['policy_arn']
    )
    versioned_arn = version_resp['policyVersionArn']
    p['versioned_arn'] = versioned_arn
    versioned_arns.append(versioned_arn)
    print(f"Versioned: {p['name']} -> {versioned_arn}")

guardrail_name = f"ar-eval-hub-{unique_id}"
guardrail_resp = bedrock_client.create_guardrail(
    name=guardrail_name,
    description="AR evaluation hub guardrail \u2014 SF Housing Code",
    automatedReasoningPolicyConfig={
        'policies': [{'policyArn': arn} for arn in versioned_arns]
    },
    blockedInputMessaging="Input blocked.",
    blockedOutputsMessaging="Output blocked."
)

guardrail_id = guardrail_resp['guardrailId']
guardrail_version = guardrail_resp['version']
print(f"\nGuardrail created: {guardrail_id} (version {guardrail_version})")

In [ ]:
def get_finding_type(finding):
    """Extract the verdict type and detail from a union-typed AR finding."""
    for ft in FINDING_TYPES:
        if ft in finding:
            return ft, finding[ft]
    return "unknown", {}

def extract_translation_metrics(finding_type, finding_detail):
    """Extract translation quality metrics from a finding's detail object."""
    if finding_type == 'translationAmbiguous':
        options = finding_detail.get('options', [])
        confidences = []
        for opt in options:
            for t in opt.get('translations', []):
                c = t.get('confidence')
                if c is not None:
                    confidences.append(c)
        return {
            'confidence': np.mean(confidences) if confidences else None,
            'premise_translation_rate': None,
            'claim_translation_rate': None,
            'has_untranslated_claims': False,
            'has_logic_warning': False,
        }

    if finding_type in ('tooComplex', 'noTranslations'):
        return {
            'confidence': None,
            'premise_translation_rate': None,
            'claim_translation_rate': None,
            'has_untranslated_claims': False,
            'has_logic_warning': False,
        }

    translation = finding_detail.get('translation', {})
    premises = translation.get('premises', [])
    claims = translation.get('claims', [])
    untranslated_premises = translation.get('untranslatedPremises', [])
    untranslated_claims = translation.get('untranslatedClaims', [])

    total_premises = len(premises) + len(untranslated_premises)
    total_claims = len(claims) + len(untranslated_claims)

    return {
        'confidence': translation.get('confidence'),
        'premise_translation_rate': len(premises) / total_premises if total_premises > 0 else 1.0,
        'claim_translation_rate': len(claims) / total_claims if total_claims > 0 else 1.0,
        'has_untranslated_claims': len(untranslated_claims) > 0,
        'has_logic_warning': 'logicWarning' in translation,
    }

def get_aggregate_finding_type(findings):
    """Aggregate multiple findings using worst-severity ordering."""
    if not findings:
        return "noTranslations"
    worst = None
    worst_severity = float('inf')
    for finding in findings:
        ft, _ = get_finding_type(finding)
        severity = SEVERITY_ORDER.get(ft, 0)
        if severity < worst_severity:
            worst_severity = severity
            worst = ft
    return worst

def extract_rules_triggered(findings):
    """Collect rule IDs from findings."""
    rules = set()
    for finding in findings:
        ft, detail = get_finding_type(finding)
        for r in detail.get('supportingRules', []):
            rules.add(r.get('ruleId', 'unknown'))
        for r in detail.get('contradictingRules', []):
            rules.add(r.get('ruleId', 'unknown'))
    return list(rules)

print("Helper functions defined.")

## II. Live Demo — Verdict Types in Action

Let's see AR Checks in action by running hand-crafted claims through our guardrail. Each claim is designed to trigger a different verdict type.

In [ ]:
demo_claims = [
    {
        'label': 'VALID \u2014 correct showerhead rule',
        'text': 'Showerheads must have a flow rate of no more than 3 gallons per minute.',
        'expected': 'valid'
    },
    {
        'label': 'INVALID \u2014 wrong heating temperature',
        'text': 'Landlords only need to maintain heating at 60 degrees Fahrenheit.',
        'expected': 'invalid'
    },
    {
        'label': 'SATISFIABLE \u2014 ambiguous compliance',
        'text': 'My showerhead may or may not comply depending on whether it is a ball-joint type and its flow rate.',
        'expected': 'satisfiable'
    },
    {
        'label': 'NO_TRANSLATIONS \u2014 off-topic',
        'text': "The best pizza in San Francisco can be found at Tony's in North Beach.",
        'expected': 'noTranslations'
    },
    {
        'label': 'VALID \u2014 correct garage restriction',
        'text': 'Apartment building garages may only be used for the storage of automobiles.',
        'expected': 'valid'
    },
    {
        'label': 'INVALID \u2014 prohibited use',
        'text': 'You can operate a paint shop in an apartment building as long as it is in a separate room.',
        'expected': 'invalid'
    },
]

print("Running live demo claims through AR guardrail...\n")
for claim in demo_claims:
    response = bedrock_runtime_client.apply_guardrail(
        guardrailIdentifier=guardrail_id,
        guardrailVersion=guardrail_version,
        source="OUTPUT",
        content=[{"text": {"text": claim['text']}}]
    )

    findings = []
    for assessment in response.get('assessments', []):
        ar = assessment.get('automatedReasoningPolicy', {})
        findings.extend(ar.get('findings', []))

    if findings:
        ft, detail = get_finding_type(findings[0])
        confidence = None
        if ft not in ('tooComplex', 'noTranslations', 'translationAmbiguous'):
            confidence = detail.get('translation', {}).get('confidence')
        conf_str = f" (confidence: {confidence:.2f})" if confidence else ""
        match = "MATCH" if ft == claim['expected'] else "MISMATCH"
    else:
        ft = "noTranslations"
        conf_str = ""
        match = "MATCH" if ft == claim['expected'] else "MISMATCH"

    print(f"{'='*70}")
    print(f"  {claim['label']}")
    print(f"  Claim: \"{claim['text'][:80]}...\"" if len(claim['text']) > 80 else f"  Claim: \"{claim['text']}\"")
    print(f"  Verdict: {ft}{conf_str}  [{match}]")
    print()

## III. Mini Evaluation

We run 28 essential test cases through the guardrail and compute core metrics. These test cases span 6 categories:

- **validation_correctness** (10): Known valid/invalid claims across key rules
- **boundary_value** (5): Numeric thresholds at exact boundary (plus/minus 1)
- **adversarial_negation** (4): Claims with negated rules — verdict must flip
- **translation_quality** (3): Abbreviations, colloquial language, unit conversions
- **domain_coverage** (4): Claims from different housing code chapters
- **adversarial_paraphrase** (2): Same claim in different phrasings

For the full 83-case suite with all 6 metric families and 8 visualizations, see notebook `04-02-07b`.

In [ ]:
with open('data/ar_tests_essential.json') as f:
    test_cases = json.load(f)

df_tests = pd.DataFrame(test_cases)
print(f"Loaded {len(test_cases)} essential test cases\n")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_tests['test_category'].value_counts().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Tests by Category')
axes[0].set_xlabel('Count')
df_tests['expected_finding_type'].value_counts().plot.barh(ax=axes[1], color='coral')
axes[1].set_title('Tests by Expected Verdict')
axes[1].set_xlabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
def run_ar_evaluation(test_cases, guardrail_id, guardrail_version):
    """Run test cases through the AR guardrail and collect results."""
    results = []
    total = len(test_cases)

    for i, test in enumerate(test_cases):
        if (i + 1) % 10 == 0 or i == 0:
            print(f"Processing test {i+1}/{total}...")

        result = {
            'test_number': test['test_number'],
            'test_category': test['test_category'],
            'housing_code_section': test['housing_code_section'],
            'rule_domain': test['rule_domain'],
            'expected_finding_type': test['expected_finding_type'],
            'difficulty': test['difficulty'],
            'input_length': len(test['llm_output']),
        }

        try:
            start_time = time.time()
            response = bedrock_runtime_client.apply_guardrail(
                guardrailIdentifier=guardrail_id,
                guardrailVersion=guardrail_version,
                source="OUTPUT",
                content=[{"text": {"text": test['llm_output']}}]
            )
            latency_ms = (time.time() - start_time) * 1000

            findings = []
            for assessment in response.get('assessments', []):
                ar = assessment.get('automatedReasoningPolicy', {})
                findings.extend(ar.get('findings', []))

            actual_type = get_aggregate_finding_type(findings)
            result['actual_finding_type'] = actual_type
            result['finding_correct'] = (actual_type == test['expected_finding_type'])
            result['num_findings'] = len(findings)
            result['latency_ms'] = latency_ms
            result['rules_triggered'] = extract_rules_triggered(findings)
            result['api_error'] = None

            confidences = []
            has_untranslated = False
            for finding in findings:
                ft, detail = get_finding_type(finding)
                metrics = extract_translation_metrics(ft, detail)
                if metrics['confidence'] is not None:
                    confidences.append(metrics['confidence'])
                if metrics.get('has_untranslated_claims'):
                    has_untranslated = True

            result['avg_confidence'] = np.mean(confidences) if confidences else None
            result['has_untranslated_claims'] = has_untranslated

        except Exception as e:
            result['actual_finding_type'] = 'error'
            result['finding_correct'] = False
            result['api_error'] = str(e)
            result['latency_ms'] = None
            result['avg_confidence'] = None
            result['has_untranslated_claims'] = False

        results.append(result)

    return results

print("Starting AR evaluation...\n")
results = run_ar_evaluation(test_cases, guardrail_id, guardrail_version)
df = pd.DataFrame(results)

accuracy = df['finding_correct'].mean()
api_errors = df['api_error'].notna().sum()
print(f"\n{'='*60}")
print(f"Evaluation complete: {len(results)} tests")
print(f"Overall accuracy: {accuracy:.1%}")
print(f"API errors: {api_errors}")

In [ ]:
def compute_core_metrics(df):
    """Compute essential metrics for the hub evaluation."""
    valid_df = df[df['api_error'].isna()]
    accuracy = valid_df['finding_correct'].mean()

    labels = [ft for ft in FINDING_TYPES
              if ft in valid_df['expected_finding_type'].values
              or ft in valid_df['actual_finding_type'].values]

    cm = pd.crosstab(valid_df['expected_finding_type'], valid_df['actual_finding_type'],
                      dropna=False).reindex(index=labels, columns=labels, fill_value=0)

    per_type = {}
    for ft in labels:
        tp = cm.loc[ft, ft] if ft in cm.index and ft in cm.columns else 0
        fp = cm[ft].sum() - tp if ft in cm.columns else 0
        fn = cm.loc[ft].sum() - tp if ft in cm.index else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        per_type[ft] = {'precision': precision, 'recall': recall, 'f1': f1}

    types_with_data = [ft for ft in labels if ft in cm.index and cm.loc[ft].sum() > 0]
    macro_f1 = np.mean([per_type[ft]['f1'] for ft in types_with_data]) if types_with_data else 0

    expected_non_valid = valid_df[valid_df['expected_finding_type'] != 'valid']
    false_valid = expected_non_valid[expected_non_valid['actual_finding_type'] == 'valid']
    false_valid_rate = len(false_valid) / len(expected_non_valid) if len(expected_non_valid) > 0 else 0

    confidences = valid_df['avg_confidence'].dropna()
    mean_confidence = confidences.mean() if len(confidences) > 0 else None

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'false_valid_rate': false_valid_rate,
        'false_valid_count': len(false_valid),
        'mean_confidence': mean_confidence,
        'confusion_matrix': cm,
        'per_type': per_type,
        'n_evaluated': len(valid_df),
    }

metrics = compute_core_metrics(df)

print(f"{'='*60}")
print(f"  Core Metrics")
print(f"{'='*60}")
print(f"  Overall Accuracy:    {metrics['accuracy']:.1%}")
print(f"  Macro F1:            {metrics['macro_f1']:.3f}")
print(f"  Mean Confidence:     {metrics['mean_confidence']:.3f}" if metrics['mean_confidence'] else "  Mean Confidence:     N/A")
print()
if metrics['false_valid_rate'] > 0:
    print(f"  *** FALSE VALID RATE: {metrics['false_valid_rate']:.1%} ({metrics['false_valid_count']} cases) ***")
    print(f"  *** This is the most dangerous failure mode \u2014 invalid claims certified as valid ***")
else:
    print(f"  False VALID Rate:    {metrics['false_valid_rate']:.1%} (target: 0%)")
print()
print(f"  Per-Type Performance:")
for ft, m in metrics['per_type'].items():
    print(f"    {ft:25s}  P={m['precision']:.2f}  R={m['recall']:.2f}  F1={m['f1']:.2f}")

## IV. Visualizations

In [ ]:
cm = metrics['confusion_matrix']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=cm.columns, yticklabels=cm.index)
ax.set_xlabel('Actual Verdict')
ax.set_ylabel('Expected Verdict')
ax.set_title('AR Confusion Matrix \u2014 Expected vs. Actual Verdict Types')
plt.tight_layout()
plt.show()

In [ ]:
categories_radar = ['Accuracy', 'Macro F1', 'Confidence', 'No False Valid']
values = [
    metrics['accuracy'],
    metrics['macro_f1'],
    metrics['mean_confidence'] if metrics['mean_confidence'] else 0,
    1.0 - metrics['false_valid_rate'],
]
target = 0.8

angles = np.linspace(0, 2 * np.pi, len(categories_radar), endpoint=False).tolist()
values_plot = values + [values[0]]
angles_plot = angles + [angles[0]]
target_plot = [target] * (len(categories_radar) + 1)

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.plot(angles_plot, values_plot, 'o-', linewidth=2, label='Actual')
ax.fill(angles_plot, values_plot, alpha=0.25)
ax.plot(angles_plot, target_plot, '--', color='red', linewidth=1, alpha=0.5, label='Target (0.8)')
ax.set_xticks(angles)
ax.set_xticklabels(categories_radar)
ax.set_ylim(0, 1.0)
ax.set_title('Evaluation Dimensions')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

In [ ]:
pt = metrics['per_type']
types = [ft for ft in pt if pt[ft]['f1'] > 0 or ft in df['expected_finding_type'].values]
x = np.arange(len(types))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, [pt[ft]['precision'] for ft in types], width, label='Precision', color='steelblue')
ax.bar(x, [pt[ft]['recall'] for ft in types], width, label='Recall', color='coral')
ax.bar(x + width, [pt[ft]['f1'] for ft in types], width, label='F1', color='forestgreen')
ax.set_xticks(x)
ax.set_xticklabels(types, rotation=45, ha='right')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 by Verdict Type')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
cat_acc = df.groupby('test_category')['finding_correct'].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#d32f2f' if v < 0.6 else '#f9a825' if v < 0.8 else '#388e3c' for v in cat_acc]
cat_acc.plot.barh(ax=ax, color=colors)
ax.axvline(x=0.8, color='gray', linestyle='--', alpha=0.5, label='Target (80%)')
ax.set_xlabel('Accuracy')
ax.set_title('Accuracy by Test Category')
ax.legend()
plt.tight_layout()
plt.show()

## V. Save State & Next Steps

Saving evaluation state so spoke notebooks can resume without re-creating policies.

In [ ]:
checkpoint = {
    'guardrail_id': guardrail_id,
    'guardrail_version': guardrail_version,
    'policies': policies,
    'versioned_arns': versioned_arns,
    'unique_id': unique_id,
    'test_results': results,
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}

with open('data/ar_eval_state.json', 'w') as f:
    json.dump(checkpoint, f, indent=2, default=str)

print("Checkpoint saved to data/ar_eval_state.json")
print(f"  Guardrail: {guardrail_id} (version {guardrail_version})")
print(f"  Policies: {len(policies)}")
print(f"  Test results: {len(results)}")

## Where to Go Next

| Notebook | What You'll Learn | Time | Dependencies |
|----------|------------------|------|-------------|
| **04-02-07a** \u2014 Document Preprocessing | How the structured rules were generated from raw PDF. Covers chunking, filtering, anti-hallucination prompt design, and 7 failure modes. | ~30 min | Standalone |
| **04-02-07b** \u2014 Comprehensive Evaluation | Full 83-case test suite, all 6 metric families, 8 visualizations, error analysis, and the advisory validation loop. | ~30 min | Loads state from this notebook |
| **04-02-07c** \u2014 Multi-Guardrail & Fidelity | Scale to 6 domains with 3 guardrails. Iteratively optimize translation accuracy via MCP. | ~45 min | Loads state from this notebook + MCP server |

In [ ]:
SKIP_CLEANUP = True  # Set to False to delete resources

if SKIP_CLEANUP:
    print("Skipping cleanup \u2014 resources preserved for spoke notebooks.")
    print("Set SKIP_CLEANUP = False and re-run this cell to delete.")
else:
    try:
        bedrock_client.delete_guardrail(guardrailIdentifier=guardrail_id)
        print(f"Deleted guardrail: {guardrail_id}")
    except Exception as e:
        print(f"Error deleting guardrail: {e}")

    for p in policies:
        try:
            bedrock_client.delete_automated_reasoning_policy(
                policyArn=p['policy_arn']
            )
            print(f"Deleted policy: {p['name']}")
        except Exception as e:
            print(f"Error deleting policy {p['name']}: {e}")

    print("\nCleanup complete.")